# 1 â€” EDA v2 (ID-targeted: no more 'not in sample')

Why v2: entity_ids are random hashes, so nrows=head samples never overlap gt ids.
Fix: chunk-scan full files (memory-safe), then index ONLY ids we need.

In [ ]:
%pip install -q pandas numpy matplotlib rapidfuzz tqdm
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/AMAZON_ML'
TRAIN = f'{BASE}/data/student_resource/dataset/train'
TEST  = f'{BASE}/data/student_resource/dataset/test'
WORK  = f'{BASE}/work'
print('ok')

In [ ]:
# A. Chunk-scan FULL files: row counts + null rates + country mix (no full load)
import pandas as pd
files = {'train_s1': f'{TRAIN}/train_source1.tsv','train_s2': f'{TRAIN}/train_source2.tsv','train_s3': f'{TRAIN}/train_source3.tsv','test_s1': f'{TEST}/test_source1.tsv','test_s2': f'{TEST}/test_source2.tsv','test_s3': f'{TEST}/test_source3.tsv'}
for name, p in files.items():
    n=no=na=nc=0; co={}
    for ch in pd.read_csv(p, sep='\t', chunksize=200000, dtype=str, keep_default_na=True):
        n+=len(ch); no+=ch['business_name'].isna().sum(); na+=ch['business_address'].isna().sum(); nc+=ch['country'].isna().sum()
        for k,v in ch['country'].value_counts(dropna=False).items(): co[str(k)]=co.get(str(k),0)+int(v)
    print(f'{name}: rows={n:,} null_name={no:,} null_addr={na:,} null_cty={nc:,} countries={co}')

In [ ]:
# # B. Ground-truth FULL stats (chunked): match-count dist, singleton rate, S2 vs S3 links
import pandas as pd
from collections import Counter
c=Counter(); n=0; ns2=ns3=0; gr=0
for ch in pd.read_csv(f'{TRAIN}/train_ground_truth.tsv', sep='\t', chunksize=200000, dtype=str, keep_default_na=False):
    gr+=len(ch)
    for m in ch['matched_entity_ids']:
        m=str(m).strip()
        ids=[x.strip() for x in m.split(',') if x.strip()] if m else []
        c[len(ids)]+=1; n+=1
        ns2+=sum(1 for x in ids if x.startswith('S2-')); ns3+=sum(1 for x in ids if x.startswith('S3-'))
print('gt_rows:',gr)
print('match-count dist:',dict(sorted(c.items())))
print(f'singleton rate: {c[0]/n:.3f} | S2 links={ns2:,} S3 links={ns3:,}')


In [ ]:
# # C. REAL eyeball pairs: index ONLY the ids we need (chunk-scan, memory-safe)
import pandas as pd
gt=pd.read_csv(f'{TRAIN}/train_ground_truth.tsv', sep='\t', nrows=15, dtype=str, keep_default_na=False)
need1=set(gt['source1_entity_id']); need23=set()
for m in gt['matched_entity_ids']:
    [need23.add(x.strip()) for x in str(m).split(',')[:4] if x.strip()]
print(f'need {len(need1)} S1 + {len(need23)} S2/S3')
idx={}
jobs=[(f'{TRAIN}/train_source1.tsv',need1),(f'{TRAIN}/train_source2.tsv',need23),(f'{TRAIN}/train_source3.tsv',need23)]
for pth,need in jobs:
    done=set()
    for ch in pd.read_csv(pth, sep='\t', chunksize=200000, dtype=str, keep_default_na=True):
        hit=ch[ch['entity_id'].isin(need)]
        for _,r in hit.iterrows(): idx[r['entity_id']]=r.to_dict()
        done={k for k in need if k in idx}
        if done==need: break
print(f"resolved {len([k for k in need1 if k in idx])}/{len(need1)} S1, {len([k for k in need23 if k in idx])}/{len(need23)} S2/S3")
for _,row in gt.head(15).iterrows():
    print('='*110)
    print('S1:',row['source1_entity_id'],'|',idx.get(row['source1_entity_id'],'NOT-FOUND'))
    [print('  MTCH:',m.strip(),'|',idx.get(m.strip(),'NOT-FOUND')) for m in str(row['matched_entity_ids']).split(',')[:4] if m.strip()]


In [ ]:
# # D. True-pair similarity sanity (sets blocking expectations)
import pandas as pd, statistics
from rapidfuzz.fuzz import token_set_ratio
gt=pd.read_csv(f'{TRAIN}/train_ground_truth.tsv', sep='\t', nrows=60, dtype=str, keep_default_na=False)
need1=set(gt['source1_entity_id']); need23=set(); pairs=[]
for _,r in gt.iterrows():
    ids=[x.strip() for x in str(r['matched_entity_ids']).split(',') if x.strip()]
    if ids: pairs.append((r['source1_entity_id'],ids[0])); need23.add(ids[0])
idx={}
for pth,need in [(f'{TRAIN}/train_source1.tsv',need1),(f'{TRAIN}/train_source2.tsv',need23),(f'{TRAIN}/train_source3.tsv',need23)]:
    for ch in pd.read_csv(pth, sep='\t', chunksize=200000, dtype=str, keep_default_na=True):
        [idx.__setitem__(r['entity_id'],r.to_dict()) for _,r in ch[ch['entity_id'].isin(need)].iterrows()]
for a,b in pairs:
    ok=a in idx and b in idx
    s1=token_set_ratio(str(idx[a].get('business_name') or ''),str(idx[b].get('business_name') or '')) if ok else None
    s2=token_set_ratio(str(idx[a].get('business_address') or ''),str(idx[b].get('business_address') or '')) if ok else None
    print(f'{a} <-> {b} | name_sim={st1} addr_sim={st2}')
    na=str(idx[a].get('business_name') or '') if a in idx else '?'; nb=str(idx[b].get('business_name') or '') if b in idx else '?'
    print(f'  {na[:70]!r} <-> {nb[:70]!r}')
print(f'n={len(ns)} name_med={statistics.median(ns):.1f} name_mean={statistics.mean(ns):.1f} addr_med={statistics.median(ad):.1f} addr_mean={statistics.mean(ad):.1f}')
ns=[]; ad=[]; sh=0
for a,b in pairs:
    ok=a in idx and b in idx
    s1=token_set_ratio(str(idx[a].get('business_name') or ''),str(idx[b].get('business_name') or '')) if ok else None
    s2=token_set_ratio(str(idx[a].get('business_address') or ''),str(idx[b].get('business_address') or '')) if ok else None
    st1='nan' if s1 is None else f'{s1:.1f}'; st2='nan' if s2 is None else f'{s2:.1f}'
    tmp1=ns.append(s1) if s1 is not None else None
    tmp2=ad.append(s2) if s2 is not None else None
    print(f'{a} <-> {b} | name_sim={st1} addr_sim={st2}')
    na=str(idx[a].get('business_name') or '') if a in idx else '?'; nb=str(idx[b].get('business_name') or '') if b in idx else '?'
    print(f'  {na[:70]!r} <-> {nb[:70]!r}')
    print(f'n={len(ns)} name_med={__import__("statistics").median(ns):.1f} name_mean={__import__("statistics").mean(ns):.1f} addr_med={__import__("statistics").median(ad):.1f} addr_mean={__import__("statistics").mean(ad):.1f}')
